<font color='#007CE5'>**CÓDIGO EL PERIODICO**</font>

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup
import time
import random
import pandas as pd
import re
from datetime import datetime

# CONFIGURACIÓN DRIVERS

driver_busqueda = webdriver.Chrome()
driver_noticia = webdriver.Chrome()

driver_busqueda.set_page_load_timeout(40)
driver_noticia.set_page_load_timeout(90)

driver_busqueda.get("https://www.elperiodico.cat/ca/successos/")
wait = WebDriverWait(driver_busqueda, 10)

# ACEPTAR COOKIES

try:
    cookies = driver_busqueda.find_element(By.ID, "didomi-notice-agree-button")
    cookies.click()
except:
    print("No se ha encontrado el botón de cookies o ya ha sido aceptado.")

# PARÁMETROS Y FUNCIONES

max_noticias = 5000
fecha_minima = datetime.strptime("2020-01-01", "%Y-%m-%d").date()
fecha_maxima = datetime.strptime("2025-06-30", "%Y-%m-%d").date()

def reiniciar_driver_noticia():
    global driver_noticia
    try:
        driver_noticia.quit()
    except:
        pass
    driver_noticia = webdriver.Chrome()
    driver_noticia.set_page_load_timeout(90)
    print("driver_noticia reiniciado.")

def cargar_pagina_con_reintento(driver, url, reintentos=3):
    for intento in range(reintentos):
        try:
            driver.get(url)
            return True
        except Exception as e:
            print(f"Error cargando {url}: {e} (intento {intento+1}/{reintentos})")
            time.sleep(5)
    return False

def extraer_fecha_desde_url(link):
    match = re.search(r'/(\d{8})/', link)
    if match:
        try:
            fecha_raw = match.group(1)
            fecha_obj = datetime.strptime(fecha_raw, "%Y%m%d")
            return fecha_obj.date().isoformat()
        except ValueError:
            return None
    return None

def extraer_noticias(html):
    soup = BeautifulSoup(html, 'html.parser')
    noticias = []

    for articulo in soup.find_all('article'):
        titulo_tag = articulo.find('h2') or articulo.find('h3')
        enlace_tag = articulo.find('a', href=True)

        if titulo_tag and enlace_tag:
            titulo = titulo_tag.get_text(strip=True)
            link = enlace_tag['href']

            if link.startswith("/"):
                link = "https://www.elperiodico.cat/" + link
            link = link.split("?")[0].split("#")[0].strip()

            fecha = None
            match = re.search(r'/(\d{4})-(\d{2})-(\d{2})/', link)
            if match:
                anio, mes, dia = match.groups()
                fecha = f"{anio}-{mes}-{dia}"

            noticias.append((titulo, link, fecha))

    return noticias

def extraer_parrafos_y_fecha(driver, url):
    if not cargar_pagina_con_reintento(driver, url):
        reiniciar_driver_noticia()
        if not cargar_pagina_con_reintento(driver, url):
            return "", None

    time.sleep(random.uniform(2.5, 5.5))

    try:

        try:
            boton_cookies = driver.find_element(By.ID, "didomi-notice-agree-button")
            boton_cookies.click()
            time.sleep(1)
        except:
            pass

        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "p")))

        soup = BeautifulSoup(driver.page_source, 'html.parser')
        parrafos = soup.find_all("p", class_="paragraph") or soup.find_all("p")

        parrafos_significativos = []
        for p in parrafos:
            texto = p.get_text(strip=True)
            if len(texto) > 40:
                parrafos_significativos.append(texto)
            if len(parrafos_significativos) >= 2:
                break

        texto_final = " ".join(parrafos_significativos)
        fecha = extraer_fecha_desde_url(url)

        return texto_final, fecha

    except Exception as e:
        print(f"Error al extraer párrafos: {e} - URL: {url}")
        return "", None

# VARIABLES DE CONTROL

todos_los_resultados = []
urls_vistas = set()
fin_scraping = False

df_init = pd.DataFrame(columns=["Título", "Enlace", "Fecha", "Primeros_Párrafos"])
df_init.to_csv("noticias_elperiodico.csv", index=False, encoding="utf-8-sig")

# BUCLE PRINCIPAL

while True:
    time.sleep(2)
    html = driver_busqueda.page_source
    nuevos = extraer_noticias(html)
    print(f"Se han encontrado {len(nuevos)} noticias en esta página.")

    nuevos_filtrados = []
    for titulo, link, fecha in nuevos:
        link = link.split("?")[0].split("#")[0].strip()
        if link not in urls_vistas:
            urls_vistas.add(link)
            nuevos_filtrados.append((titulo, link, fecha))

    print(f"Noticias nuevas para procesar: {len(nuevos_filtrados)}")

    for titulo, link, fecha in nuevos_filtrados:
        print(f"Procesando: {titulo} - {link}")
        if len(todos_los_resultados) >= max_noticias:
            break

        time.sleep(random.uniform(2.5, 6.0))
        parrafos, fecha_real = extraer_parrafos_y_fecha(driver_noticia, link)

        print(f"Fecha extraída: {fecha_real}")

        if fecha_real:
            try:
                fecha_obj = datetime.strptime(fecha_real, "%Y-%m-%d").date()
                if not (fecha_minima <= fecha_obj <= fecha_maxima):
                    print("Noticia descartada por fecha fuera de rango.")
                    if fecha_obj < fecha_minima:
                        print("Fecha mínima alcanzada, fin del scraping.")
                        fin_scraping = True
                        break
                    continue
            except:
                print("Fecha con formato incorrecto.")
                continue
        else:
            print("Noticia descartada por no tener fecha.")
            continue

        todos_los_resultados.append((titulo, link, fecha_real, parrafos))
        print(f"{len(todos_los_resultados)}. {fecha_real} - {titulo}")

        df_row = pd.DataFrame([(titulo, link, fecha_real, parrafos)],
                            columns=["Título", "Enlace", "Fecha", "Primeros_Párrafos"])
        df_row.to_csv("noticias_elperiodico.csv", index=False, encoding="utf-8-sig", mode='a', header=False)

    if len(todos_los_resultados) >= max_noticias or fin_scraping:
        print("Fin del scraping por límite o fecha mínima alcanzados.")
        break

    driver_busqueda.execute_script("window.scrollBy(0, 1500);")
    time.sleep(1)

    try:
        boton = WebDriverWait(driver_busqueda, 5).until(
            EC.element_to_be_clickable((By.CLASS_NAME, "cargarmas"))
        )
        driver_busqueda.execute_script("arguments[0].scrollIntoView({block: 'center'});", boton)
        time.sleep(random.uniform(1.0, 2.0))
        driver_busqueda.execute_script("arguments[0].click();", boton)
        print("Clic en 'Carregar més'")
        time.sleep(random.uniform(2.0, 4.0))
    except Exception as e:
        print("No se ha podido hacer clic en el botón:", e)
        break

driver_busqueda.quit()
driver_noticia.quit()

<font color='#007CE5'>**CÓDIGO LA RAZÓN**</font>

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup

import time
import random
import pandas as pd
import re
from datetime import datetime

# CONFIGURACIÓN DRIVERS

driver_busqueda = webdriver.Chrome()
driver_noticia = webdriver.Chrome()

driver_busqueda.set_page_load_timeout(50)
driver_noticia.set_page_load_timeout(90)

driver_busqueda.get("https://www.larazon.es/cataluna/")
wait = WebDriverWait(driver_busqueda, 20)

# ACEPTAR COOKIES

try:
    cookies = driver_busqueda.find_element(By.ID, "didomi-notice-agree-button")
    cookies.click()
except:
    print("No se ha encontrado el botón de cookies o ya ha sido aceptado.")

# PARÁMETROS, ETIQUETAS, FUNCIONES

max_noticias = 5000
fecha_minima = datetime.strptime("2020-01-01", "%Y-%m-%d").date()
fecha_maxima = datetime.strptime("2025-06-30", "%Y-%m-%d").date()

TOPICS_VALIDOS = {"Sucesos", "Crimen organizado", "Pornografía infantil", "Okupas", "Mossos d'Esquadra", "Narcotráfico", "robos", 
                  "Delito sexual","abusossexuales", "abusosamenores", "violencia", "mossosdesquadra"}


def reiniciar_driver_noticia():
    global driver_noticia
    try:
        driver_noticia.quit()
    except:
        pass
    driver_noticia = webdriver.Chrome()
    driver_noticia.set_page_load_timeout(90)
    print("driver_noticia reiniciado.")

def cargar_pagina_con_reintento(driver, url, reintentos=3):
    for intento in range(reintentos):
        try:
            driver.get(url)
            return True
        except Exception as e:
            print(f"Error cargando {url}: {e} (intento {intento+1}/{reintentos})")
            time.sleep(5)
    return False

def extraer_fecha_desde_url(link):
    match = re.search(r'_(\d{8}).*\.html', link)
    if match:
        try:
            return datetime.strptime(match.group(1), "%Y%m%d").date().isoformat()
        except ValueError:
            pass

    match = re.search(r'/(\d{8})/', link)
    if match:
        try:
            return datetime.strptime(match.group(1), "%Y%m%d").date().isoformat()
        except ValueError:
            pass

    return None

def extraer_noticias(html):
    soup = BeautifulSoup(html, 'html.parser')
    noticias = []

    titulos = soup.find_all('h2', class_='article__title')
    for h2 in titulos:
        enlace_tag = h2.find('a', href=True)
        if enlace_tag:
            link = enlace_tag['href']
            titulo = enlace_tag.get_text(strip=True)
            fecha = extraer_fecha_desde_url(link)
            noticias.append((titulo, link, fecha))

    return noticias

def extraer_parrafos_y_topic(driver, url):
    if not cargar_pagina_con_reintento(driver, url):
        reiniciar_driver_noticia()
        if not cargar_pagina_con_reintento(driver, url):
            return "", None, None

    time.sleep(random.uniform(2.5, 5.5))

    try:
        try:
            boton_cookies = driver.find_element(By.ID, "didomi-notice-agree-button")
            boton_cookies.click()
            time.sleep(1)
        except:
            pass

        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "p")))

        soup = BeautifulSoup(driver.page_source, 'html.parser')
        contenedor_texto = soup.select_one("main.article") or soup.select_one("div.c-article-content") or soup
        parrafos = contenedor_texto.find_all("p")
        textos_fuera = ["la tienda de la razón", "lea el periódico", "leer más", "descarga el pdf", "suscríbete", "publicidad", "copyright"]

        parrafos_significativos = []
        for p in parrafos:
            texto = p.get_text(separator=" ", strip=True)
            texto_limpio = re.sub(r'\s+', ' ', texto).lower()

            if (len(texto_limpio) > 50 and
                not any(basura in texto_limpio for basura in textos_fuera)):
                parrafos_significativos.append(texto.strip())

            if len(parrafos_significativos) >= 3:
                break

        texto_final = " ".join(parrafos_significativos)

        fecha = extraer_fecha_desde_url(url)

        topics_encontrados = []
        tags = soup.select("a.tags-list__link")
        for tag in tags:
            topic = tag.get_text(strip=True)
            if topic in TOPICS_VALIDOS:
                topics_encontrados.append(topic)

        if not topics_encontrados:
            return "", None, None  

        return texto_final, fecha, ", ".join(topics_encontrados)

    except Exception as e:
        print(f"Error al extraer párrafos: {e} - URL: {url}")
        return "", None, None


# VARIABLES DE CONTROL

todos_los_resultados = []
urls_vistas = set()
fin_scraping = False

df_init = pd.DataFrame(columns=["Título", "Enlace", "Fecha", "Primeros_Párrafos", "Topic"])
df_init.to_csv("noticias_larazon.csv", index=False, encoding="utf-8-sig")


# BUCLE PRINCIPAL

while True:
    time.sleep(2)
    html = driver_busqueda.page_source
    nuevos = extraer_noticias(html)
    print(f"Se han encontrado {len(nuevos)} noticias en esta página.")

    nuevos_filtrados = []
    for titulo, link, fecha in nuevos:
        link = link.split("?")[0].split("#")[0].strip()
        if link not in urls_vistas:
            urls_vistas.add(link)
            nuevos_filtrados.append((titulo, link, fecha))

    print(f"Noticias nuevas para procesar: {len(nuevos_filtrados)}")

    for titulo, link, fecha in nuevos_filtrados:
        print(f"Procesando: {titulo} - {link}")
        if len(todos_los_resultados) >= max_noticias:
            break

        time.sleep(random.uniform(2.5, 6.0))
        parrafos, fecha_real, topic = extraer_parrafos_y_topic(driver_noticia, link)

        print(f"Fecha extraída: {fecha_real}, Topic: {topic}")

        if not fecha_real or not topic:
            print("Noticia descartada por no tener fecha o topic válido.")
            continue

        try:
            fecha_obj = datetime.strptime(fecha_real, "%Y-%m-%d").date()
            if not (fecha_minima <= fecha_obj <= fecha_maxima):
                print("Noticia descartada por fecha fuera de rango.")
                if fecha_obj < fecha_minima:
                    print("Fecha mínima alcanzada, fin del scraping.")
                    fin_scraping = True
                    break
                continue
        except:
            print("Fecha con formato incorrecto.")
            continue

        todos_los_resultados.append((titulo, link, fecha_real, parrafos, topic))
        print(f"{len(todos_los_resultados)}. {fecha_real} - {titulo} [{topic}]")

        df_row = pd.DataFrame([(titulo, link, fecha_real, parrafos, topic)],
                            columns=["Título", "Enlace", "Fecha", "Primeros_Párrafos", "Topic"])
        df_row.to_csv("noticias_larazon.csv", index=False, encoding="utf-8-sig", mode='a', header=False)

    if len(todos_los_resultados) >= max_noticias or fin_scraping:
        print("Fin del scraping por límite o fecha mínima alcanzados.")
        break

    driver_busqueda.execute_script("window.scrollBy(0, 1500);")
    time.sleep(1)

    try:
        soup = BeautifulSoup(driver_busqueda.page_source, "html.parser")
        siguiente = soup.select_one("li.pagination__item--next a")
        if siguiente and siguiente.get("href"):
            next_url = siguiente["href"]
            driver_busqueda.get(next_url)
            time.sleep(random.uniform(2.0, 4.0))
        else:
            print("No hay enlace 'Siguiente'. Fin del scraping.")
            break
    except Exception as e:
        print(f"Error al intentar ir a la siguiente página: {e}")
        break

driver_busqueda.quit()
driver_noticia.quit()
print("CSV guardado como noticias_larazon.csv")


<font color='#007CE5'>**CÓDIGO LA VANGUARDIA**</font>

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup
import time
import random
import pandas as pd
import re
from datetime import datetime

# CONFIGURACIÓN DRIVERS

driver_busqueda = webdriver.Chrome()
driver_noticia = webdriver.Chrome()

driver_busqueda.set_page_load_timeout(40)
driver_noticia.set_page_load_timeout(80)

driver_busqueda.get("https://stories.lavanguardia.com/search?q=Catalunya&author=&category=&section=sucesos&startDate=01%2F01%2F2020&endDate=30%2F06%2F2025&sort=")
wait = WebDriverWait(driver_busqueda, 15)

# ACEPTAR COOKIES

try:
    cookies = driver_busqueda.find_element(By.ID, "didomi-notice-agree-button")
    cookies.click()
except:
    print("No se ha encontrado el botón de cookies o ya ha sido aceptado.")

# PARÁMETROS Y FUNCIONES

max_noticias = 5000
fecha_minima = datetime.strptime("2020-01-01", "%Y-%m-%d").date()
fecha_maxima = datetime.strptime("2025-06-30", "%Y-%m-%d").date()

def extraer_fecha_desde_url(link):
    match = re.search(r'/(\d{8})/', link)
    if match:
        try:
            fecha_raw = match.group(1)
            fecha_obj = datetime.strptime(fecha_raw, "%Y%m%d")
            return fecha_obj.date().isoformat()
        except ValueError:
            return None
    return None

def extraer_noticias(html):
    soup = BeautifulSoup(html, 'html.parser')
    noticias = []

    for articulo in soup.find_all('article'):
        titulo_tag = articulo.find('h2') or articulo.find('h3')
        enlace_tag = articulo.find('a', href=True)

        if titulo_tag and enlace_tag:
            titulo = titulo_tag.get_text(strip=True)
            link = enlace_tag['href']

            if link.startswith("/"):
                link = "https://stories.lavanguardia.com" + link
            link = link.split("?")[0].split("#")[0].strip()

            fecha = None
            match = re.search(r'/(\d{4})-(\d{2})-(\d{2})/', link)
            if match:
                anio, mes, dia = match.groups()
                fecha = f"{anio}-{mes}-{dia}"
            else:
                match = re.search(r'/(\d{4})-(\d{2})-(\d{2})/', link)
                if match:
                    anio, mes, dia = match.groups()
                    fecha = f"{anio}-{mes}-{dia}"
            

            noticias.append((titulo, link, fecha))

    return noticias

def extraer_parrafos_y_fecha(driver, url):
    try:
        driver.get(url)
        time.sleep(2)

        try:
            boton_cookies = driver.find_element(By.ID, "didomi-notice-agree-button")
            boton_cookies.click()
            time.sleep(1)
        except:
            pass

        try:
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, "paragraph")))
        except TimeoutException:
            print(f"Timeout esperando párrafos en: {url}")
            return "", extraer_fecha_desde_url(url)

        soup = BeautifulSoup(driver.page_source, 'html.parser')

        parrafos = soup.find_all("p", class_="paragraph")
        parrafos_significativos = []

        for p in parrafos:
            texto = p.get_text(strip=True)
            if len(texto) > 40:
                parrafos_significativos.append(texto)
            if len(parrafos_significativos) >= 2:
                break

        texto_final = " ".join(parrafos_significativos)

        match = re.search(r'/(\d{8})/', url)
        fecha = None
        if match:
            try:
                fecha_raw = match.group(1)
                fecha_obj = datetime.strptime(fecha_raw, "%Y%m%d")
                fecha = fecha_obj.date().isoformat()
            except:
                pass

        return texto_final, fecha

    except Exception as e:
        print(f"Error al extraer párrafos: {e} - URL: {url}")
        return "", None


# VARIABLES DE CONTROL

todos_los_resultados = []
urls_vistas = set()

df_init = pd.DataFrame(columns=["Título", "Enlace", "Fecha", "Primeros_Párrafos"])
df_init.to_csv("noticias_lavanguardia.csv", index=False, encoding="utf-8-sig")

# BUCLE PRINCIPAL

while True:
    time.sleep(2)
    html = driver_busqueda.page_source
    nuevos = extraer_noticias(html)
    print(f"Se han encontrado {len(nuevos)} noticias en esta página.")

    nuevos_filtrados = []
    for titulo, link, fecha in nuevos:
        link = link.split("?")[0].split("#")[0].strip()
        if link not in urls_vistas:
            urls_vistas.add(link)
            nuevos_filtrados.append((titulo, link, fecha))

    print(f"Noticias nuevas para procesar: {len(nuevos_filtrados)}")

    for titulo, link, fecha in nuevos_filtrados:
        print(f"Procesando: {titulo} - {link}")
        if len(todos_los_resultados) >= max_noticias:
            break

        time.sleep(random.uniform(1.5, 3.5))
        parrafos, fecha_real = extraer_parrafos_y_fecha(driver_noticia, link)

        print(f"Fecha extraída: {fecha_real}")

        if fecha_real:
            try:
                fecha_obj = datetime.strptime(fecha_real, "%Y-%m-%d").date()
                if not (fecha_minima <= fecha_obj <= fecha_maxima):
                    print("Noticia descartada por fecha fuera de rango.")
                    continue
            except:
                print("Fecha con formato incorrecto.")
                continue
        else:
            print("Noticia descartada por no tener fecha.")
            continue

        todos_los_resultados.append((titulo, link, fecha_real, parrafos))
        print(f"{len(todos_los_resultados)}. {fecha_real} - {titulo}")

        df_row = pd.DataFrame([(titulo, link, fecha_real, parrafos)],
                            columns=["Título", "Enlace", "Fecha", "Primeros_Párrafos"])
        df_row.to_csv("noticias_lavanguardia.csv", index=False, encoding="utf-8-sig", mode='a', header=False)


    if len(todos_los_resultados) >= max_noticias:
        print("Límite alcanzado.")
        break

    driver_busqueda.execute_script("window.scrollBy(0, 1500);")
    time.sleep(1)

    try:
        boton = WebDriverWait(driver_busqueda, 15).until(
            EC.element_to_be_clickable((By.XPATH, '//a[span[text()="Siguiente »"]]'))
        )
        driver_busqueda.execute_script("arguments[0].scrollIntoView({block: 'center'});", boton)
        time.sleep(0.5)
        driver_busqueda.execute_script("arguments[0].click();", boton)
        print("Clic en 'Siguiente'")
        time.sleep(2)
    except Exception as e:
        print("No se ha podido hacer clic en el botón:", e)
        break


driver_busqueda.quit()
driver_noticia.quit()

<font color='#007CE5'>**CÓDIGO DIARI ARA**</font>

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time, random, re, os
import pandas as pd
from datetime import datetime
from pathlib import Path

# CONFIGURACIÓN
debug_mode = True
profile_path = r"C:\Users\Usuario\Desktop\ChromeSeleniumProfile"
csv_file = "noticias_ara.csv"
debug_dir = Path("debug_html")
debug_dir.mkdir(exist_ok=True)

options = webdriver.ChromeOptions()
options.add_argument(fr"user-data-dir={profile_path}")
driver = webdriver.Chrome(options=options)
driver.set_page_load_timeout(40)
wait = WebDriverWait(driver, 15)

driver.get("https://www.ara.cat/cercador?text=successos")
time.sleep(2)


df_init = pd.DataFrame(columns=["Título", "Enlace", "Fecha", "Primeros_Párrafos", "Topic"])
df_init.to_csv(csv_file, index=False, encoding="utf-8-sig")


# FUNCIONES

def normalize_text(txt):
    if not txt:
        return ""
    return re.sub(r'\s+', ' ', txt).strip()

def normalize_lower(txt):
    return normalize_text(txt).lower()

def save_debug_html(url, html):
    safe = re.sub(r'[^0-9a-zA-Z\-_.]', '_', url)[:150]
    path = debug_dir / f"{safe}.html"
    path.write_text(html, encoding="utf-8")
    return path

def extraer_noticias(html):
    soup = BeautifulSoup(html, 'html.parser')
    noticias = []
    for articulo in soup.find_all('article'):
        titulo_tag = articulo.find('h2') or articulo.find('h3')
        enlace_tag = articulo.find('a', href=True)
        if titulo_tag and enlace_tag:
            titulo = titulo_tag.get_text(strip=True)
            link = enlace_tag['href']
            if link.startswith("/"):
                link = "https://www.ara.cat" + link
            link = link.split("?")[0].split("#")[0].strip()
            if not re.search(r'/periodista/|/autor/|/colaborador/', link):
                noticias.append((titulo, link))
    return noticias

def extraer_fecha(soup, link, driver=None):
    time_tag = soup.find('time')
    if time_tag and time_tag.has_attr('datetime'):
        try:
            return datetime.fromisoformat(time_tag['datetime']).date()
        except:
            pass

    if time_tag:
        text_time = normalize_text(time_tag.get_text())
        for fmt in ("%d/%m/%Y", "%Y-%m-%d"):
            try:
                return datetime.strptime(text_time, fmt).date()
            except:
                pass

    fecha_div = soup.find("div", class_="opening-actions__date")
    if fecha_div:
        txt = normalize_text(fecha_div.get_text())
        for fmt in ("%d/%m/%Y", "%Y-%m-%d"):
            try:
                return datetime.strptime(txt, fmt).date()
            except:
                pass

    if driver:
        try:
            meta = driver.find_element(By.CSS_SELECTOR, 'meta[property="article:published_time"]')
            content = meta.get_attribute("content")
            if content:
                return datetime.fromisoformat(content.split('+')[0]).date()
        except:
            pass

    m = re.search(r'/(\d{4})[-/](\d{2})[-/](\d{2})/', link)
    if m:
        try:
            y, mn, d = m.groups()
            return datetime(int(y), int(mn), int(d)).date()
        except:
            pass
    return None

topics_validos = {"successos", "violència masclista", "violències masclistes", "ciberseguretat", "macrooperatiu"}
topics_validos_lower = {t.lower() for t in topics_validos}

def extraer_parrafos_y_fecha(driver, url, debug=False):
    try:
        driver.execute_script("window.open('');")
        driver.switch_to.window(driver.window_handles[-1])
        driver.get(url)

        try:
            wait.until(EC.presence_of_element_located((
                By.CSS_SELECTOR,
                "div.article-body, div.opening-actions__date, div.topic, a.tag, a.tags-list__link, time"
            )), timeout=12)
        except:
            if debug:
                print("No se han encontrado selectores clave en el tiempo proporcionado.")

        time.sleep(1.0)
        page_html = driver.page_source
        soup = BeautifulSoup(page_html, "html.parser")

        candidates = []
        topic_div = soup.find("div", class_="topic")
        if topic_div:
            candidates.append(normalize_text(topic_div.get_text()))

        for a in soup.select("a.tag, a.tags-list__link"):
            txt = normalize_text(a.get_text())
            href = a.get("href", "")
            candidates.append(f"{txt}||{href}")

        meta_kw = soup.find("meta", {"name": "keywords"})
        if meta_kw and meta_kw.has_attr("content"):
            for kw in meta_kw["content"].split(","):
                candidates.append(normalize_text(kw))

        meta_section = soup.find("meta", {"property": "article:section"}) or soup.find("meta", {"name": "section"})
        if meta_section and meta_section.has_attr("content"):
            candidates.append(normalize_text(meta_section["content"]))

        cand_texts = []
        for c in candidates:
            if "||" in c:
                txt, href = c.split("||", 1)
                cand_texts.append(normalize_lower(txt))
                cand_texts.append(normalize_lower(href))
            else:
                cand_texts.append(normalize_lower(c))
        cand_texts = [ct for ct in dict.fromkeys(cand_texts) if ct]

        matched_topics = []
        for ct in cand_texts:
            for topic_ref in topics_validos_lower:
                if topic_ref in ct or ct in topic_ref:
                    matched_topics.append(topic_ref)
        matched_topics = list(dict.fromkeys(matched_topics))
        if not matched_topics:
            driver.close()
            driver.switch_to.window(driver.window_handles[0])
            return "", None, None

        chosen_topic = matched_topics[0]

        cont = soup.find('div', class_='article-body') or soup
        parrafos = cont.find_all('p')
        textos = []
        for p in parrafos:
            t = normalize_text(p.get_text())
            t_low = t.lower()
            if len(t) > 40 and "publicidad" not in t_low and "suscríbete" not in t_low:
                textos.append(t)
            if len(textos) >= 2:
                break
        texto_final = " ".join(textos)

        fecha = extraer_fecha(soup, url, driver=driver)

        driver.close()
        driver.switch_to.window(driver.window_handles[0])

        return texto_final, fecha.isoformat() if fecha else None, chosen_topic

    except Exception as e:
        print(f"Error al cargar: {e} - URL: {url}")
        try:
            driver.close()
            driver.switch_to.window(driver.window_handles[0])
        except:
            pass
        return "", None, None

def click_mostrar_mas(driver, timeout=10, debug=True):
    try:
        wait = WebDriverWait(driver, timeout)
        try:
            boton = wait.until(EC.element_to_be_clickable(
                (By.XPATH, "//button[.//span[contains(text(), \"Mostra'n més\")]]")
            ))
        except:
            boton_span = wait.until(EC.presence_of_element_located(
                (By.XPATH, "//span[contains(text(), \"Mostra'n més\")]")
            ))
            boton = boton_span.find_element(By.XPATH, "./ancestor::button")

        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", boton)
        time.sleep(0.3)
        driver.execute_script("arguments[0].click();", boton)
        if debug:
            print("Clic en 'Mostra'n més'")
        return True
    except Exception as e:
        if debug:
            print(f"No se pudo hacer clic en 'Mostra'n més': {e}")
        return False


# BUCLE PRINCIPAL

urls_vistas = set()
contador = 0
max_noticias = 5000
fecha_minima = datetime.strptime("2020-01-01", "%Y-%m-%d").date()
fecha_maxima = datetime.strptime("2025-06-30", "%Y-%m-%d").date()

while True:
    time.sleep(2)
    html = driver.page_source
    nuevos = extraer_noticias(html)
    if debug_mode:
        print(f"\nPágina: se encontraron {len(nuevos)} artículos listados.")
    nuevos_filtrados = []
    for titulo, link in nuevos:
        link = link.split("?")[0].split("#")[0].strip()
        if link not in urls_vistas:
            urls_vistas.add(link)
            nuevos_filtrados.append((titulo, link))

    for titulo, link in nuevos_filtrados:
        if contador >= max_noticias:
            break
        print(f"Procesando: {titulo} - {link}")
        time.sleep(random.uniform(1.2, 2.5))
        parrafos, fecha_real, topic = extraer_parrafos_y_fecha(driver, link, debug=debug_mode)
        if not fecha_real:
            continue
        fecha_obj = datetime.strptime(fecha_real, "%Y-%m-%d").date()
        if not (fecha_minima <= fecha_obj <= fecha_maxima):
            continue

        df_temp = pd.DataFrame([(titulo, link, fecha_real, parrafos, topic)],
                               columns=["Título", "Enlace", "Fecha", "Primeros_Párrafos", "Topic"])
        df_temp.to_csv(csv_file, mode="a", header=False, index=False, encoding="utf-8-sig")
        contador += 1
        print(f"  Guardado ({contador}): {fecha_real} - {titulo} [{topic}]")

    if not click_mostrar_mas(driver, debug=True):
        print("➡️ Fin de resultados.")
        break
    time.sleep(1.6)

driver.quit()
